# Исследование данных NSD — интерактивный блокнот

Этот блокнот для того, чтобы **щупать данные руками**. В отличие от
`01_data_sanity.ipynb`, который проверяет корректность, здесь цель — понять, что вообще
внутри, и задавать свои вопросы.

**Как пользоваться.** В начале каждого раздела есть блок «РУЧКИ» — переменные заглавными
буквами. Меняй их и перезапускай ячейку. Всё остальное подстроится.

**Что где лежит:**

| Путь | Что это |
|---|---|
| `data/stimuli/shared1000_images.npy` | 1000 картинок, которые видели все испытуемые |
| `data/betas/subjXX_shared_betas.h5` | ответы мозга: трайлы × вершины коры |
| `data/meta/nsd_expdesign.mat` | когда какую картинку показывали |
| `data/meta/subjXX/*.ncsnr.mgh` | оценка сигнал/шум на каждую вершину от авторов NSD |
| `cache/activations/*.h5` | активации зрительных энкодеров |
| `cache/vlm_activations/*.h5` | активации всего стека VLM |

**Про память.** Один испытуемый — это 812 MB в оперативке. Не держи несколько сразу:
после работы делай `del subj`.

## 0. Настройка

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Ищем корень репозитория вверх по дереву — блокнот заработает из любой папки.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from nsd_rsa.design import load_expdesign, shared_trials, usable_images
from nsd_rsa.loaders import (
    average_repeats,
    common_valid_vertices,
    load_subject,
    split_half,
)
from nsd_rsa.noise_ceiling import split_half_reliability
from nsd_rsa.rdm import compare_rdms, compute_rdm
from nsd_rsa.rois import STREAM_LABELS, load_streams

plt.rcParams["figure.dpi"] = 110
print("корень репозитория:", ROOT)

SUBJECTS = [f"subj{i:02d}" for i in range(1, 9)]
DESIGN = load_expdesign(ROOT / "data/meta/nsd_expdesign.mat")

# 515 картинок с тремя повторами у всех восьмерых — основной аналитический набор.
IMAGES = usable_images(SUBJECTS, DESIGN, min_repeats=3)
print(f"аналитическое подмножество: {len(IMAGES)} картинок из 1000")

# Вершины, у которых есть данные у ВСЕХ испытуемых (у subj06/08 часть V1 вне среза).
VALID = common_valid_vertices(ROOT / "data/betas", ROOT / "cache/valid_vertices.npy")
print(f"валидных вершин: {VALID.sum():,} из {len(VALID):,}")

In [ ]:
def show_images(idx, titles=None, cols=4, size=2.2, suptitle=None):
    """Показать картинки по их номерам-слотам (0..999)."""
    idx = np.atleast_1d(idx)
    rows = int(np.ceil(len(idx) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for k, i in enumerate(idx):
        ax = np.atleast_1d(axes).ravel()[k]
        ax.imshow(STIMULI[i])
        ax.set_title(titles[k] if titles is not None else f"#{i}", fontsize=8)
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    fig.tight_layout()
    plt.show()


STIMULI = np.load(ROOT / "data/stimuli/shared1000_images.npy", mmap_mode="r")
print("стимулы:", STIMULI.shape, STIMULI.dtype)

## 1. Что вообще видели люди

Картинки из COCO, обрезанные до квадрата 425×425. Каждую показывали на 3 секунды.

**Ручка:** `LOOK_AT` — какие слоты смотреть. Слот это номер от 0 до 999 внутри
shared1000, не тот же самый, что id картинки в NSD.

In [ ]:
# ============ РУЧКИ ============
LOOK_AT = np.arange(0, 12)          # попробуй: np.random.choice(1000, 12, replace=False)
# ================================

show_images(LOOK_AT, suptitle="Стимулы NSD (слоты shared1000)")

In [ ]:
# Из 1000 в основной анализ идут 515. Посмотрим, что выпало и почему.
dropped = np.setdiff1d(np.arange(1000), IMAGES)
print(f"в анализе: {len(IMAGES)},  выпало: {len(dropped)}")
print("\nВыпали не по содержанию, а по дизайну: короткие испытуемые не дошли")
print("до конца программы, поэтому у части картинок меньше трёх повторов.\n")

show_images(dropped[:8], suptitle="Примеры выпавших картинок — ничем не отличаются")

## 2. Когда что показывали

`masterordering` — вектор из 30 000 чисел: для каждого трайла номер картинки. Отсюда мы
знаем, какой замер мозга какой картинке принадлежит. Ошибка здесь испортила бы всё
незаметно, поэтому в коде стоят проверки.

**Ручка:** `IMAGE_SLOT` — картинка, для которой смотрим историю показов.

In [ ]:
# ============ РУЧКИ ============
IMAGE_SLOT = 0
WHO = "subj01"
# ================================

trials = shared_trials(WHO, DESIGN)
sel = trials.image == IMAGE_SLOT

print(f"{WHO}, картинка-слот {IMAGE_SLOT} — все её показы:\n")
print(f"{'повтор':>7}{'сессия':>8}{'трайл в сессии':>16}")
for rep, sess, frame in zip(trials.repeat[sel], trials.session[sel], trials.frame[sel]):
    print(f"{rep:>7}{sess:>8}{frame:>16}")

print(f"\nВсего трайлов у {WHO}: {len(trials)}")
print("Повторы специально разнесены по разным сессиям — иначе они делили бы")
print("общий шум сканера, и оценка надёжности была бы завышена.")
show_images([IMAGE_SLOT], cols=1, size=3)

In [ ]:
# Сколько повторов доступно у каждого испытуемого — вот откуда взялись 515.
print(f"{'испытуемый':<12}{'трайлов':>9}{'картинок с 3 повт.':>20}{'с 0 повт.':>11}")
for s in SUBJECTS:
    t = shared_trials(s, DESIGN)
    reps = t.repeats_per_image()
    print(f"{s:<12}{len(t):>9}{(reps == 3).sum():>20}{(reps == 0).sum():>11}")

## 3. Мозг: матрица бет

**Бета** — оценка отклика одной вершины коры на одно предъявление одной картинки,
в процентах сигнального изменения. Матрица `(трайлы × вершины)` устроена ровно как
активации слоя сети на батче: строка = стимул, столбец = «признак».

**Ручка:** `SUBJECT`. Загрузка занимает несколько секунд и 812 MB.

In [ ]:
# ============ РУЧКИ ============
SUBJECT = "subj01"
# ================================

subj = load_subject(ROOT / f"data/betas/{SUBJECT}_shared_betas.h5")
print(f"{subj.subject}: беты {subj.betas.shape}  ({subj.betas.nbytes / 1e6:.0f} MB)")
print(f"уникальных картинок: {len(np.unique(subj.image))}\n")

print("вершин в каждой зоне:")
for roi, n in subj.roi_counts().items():
    print(f"  {roi:<12}{n:>7,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

v = subj.betas[::20].ravel()
v = v[np.isfinite(v)]
axes[0].hist(v, bins=200, range=(-10, 10), density=True, color="#356")
axes[0].set_xlabel("beta (% сигнального изменения)")
axes[0].set_title("Распределение одиночных бет")
axes[0].axvline(0, color="k", lw=0.5, ls=":")

for roi in ["early", "ventral", "lateral"]:
    m = subj.roi_mask(roi)
    s = subj.betas[::40][:, m].ravel()
    axes[1].hist(s[np.isfinite(s)], bins=150, range=(-8, 8), density=True,
                 histtype="step", label=roi, lw=1.4)
axes[1].legend(fontsize=8)
axes[1].set_xlabel("beta")
axes[1].set_title("По зонам")
plt.tight_layout()
plt.show()

print(f"среднее {v.mean():+.3f}, std {v.std():.3f}")
print("Шум огромный: одиночный замер почти бесполезен, поэтому усредняем 3 повтора.")

## 4. Одна вершина крупным планом

Здесь начинается самое интересное. Возьмём **одну точку коры** и посмотрим, на какие
картинки она откликается сильнее всего, а на какие — слабее.

Сначала находим надёжные вершины: делим 3 повтора на две непересекающиеся половины и
коррелируем. Вершина с надёжностью около нуля не несёт воспроизводимого сигнала — на неё
смотреть бессмысленно.

**Ручки:** `ROI` (зона коры) и `VERTEX_RANK` (0 = самая надёжная вершина зоны).

In [ ]:
# ============ РУЧКИ ============
ROI = "ventral"        # early | midventral | ventral | midlateral | lateral | midparietal | parietal
VERTEX_RANK = 0        # 0 = самая надёжная; попробуй 1, 5, 100
# ================================

half_a, half_b, _ = split_half(subj, images=IMAGES, roi=ROI, seed=0, valid=VALID)
reliability = split_half_reliability(half_a, half_b)

order = np.argsort(-np.nan_to_num(reliability))
v = order[VERTEX_RANK]

print(f"{ROI}: {len(reliability):,} вершин")
print(f"средняя надёжность {np.nanmean(reliability):+.3f}")
print(f"вершина ранга {VERTEX_RANK}: надёжность {reliability[v]:+.3f}")

plt.figure(figsize=(6, 2.6))
plt.hist(reliability[np.isfinite(reliability)], bins=60, color="#356")
plt.axvline(reliability[v], color="crimson", lw=2, label=f"наша вершина ({reliability[v]:+.2f})")
plt.xlabel("split-half надёжность"); plt.legend(fontsize=8)
plt.title(f"{ROI}: сколько сигнала в каждой вершине")
plt.tight_layout(); plt.show()

In [ ]:
# Ответ выбранной вершины на каждую из 515 картинок, усреднённый по повторам.
patterns, imgs = average_repeats(subj, images=IMAGES, roi=ROI, valid=VALID)
response = patterns[:, v]

rank = np.argsort(-response)
print(f"вершина ранга {VERTEX_RANK} в {ROI} у {SUBJECT}")
print(f"диапазон отклика: {response.min():+.2f} .. {response.max():+.2f}\n")

show_images(imgs[rank[:8]],
            titles=[f"{response[i]:+.2f}" for i in rank[:8]],
            suptitle="СИЛЬНЕЕ ВСЕГО откликается на это")

show_images(imgs[rank[-8:]],
            titles=[f"{response[i]:+.2f}" for i in rank[-8:]],
            suptitle="СЛАБЕЕ ВСЕГО — на это")

Посмотри на две подборки. Есть ли что-то общее внутри каждой? В вентральной коре часто
видны лица или крупные объекты сверху и пустые сцены снизу; в `early` разница скорее по
контрасту и пространственной частоте, чем по смыслу.

Это и есть содержательный смысл «избирательности» участка коры — но осторожно: одна
вершина шумная, и 8 картинок это мало. Ниже — то же самое, но усреднённо по многим
вершинам, что гораздо надёжнее.

In [ ]:
# ============ РУЧКИ ============
TOP_N_VERTICES = 200     # усредняем по стольким самым надёжным вершинам зоны
# ================================

top = order[:TOP_N_VERTICES]
group_response = patterns[:, top].mean(axis=1)
rank_g = np.argsort(-group_response)

print(f"средний отклик {TOP_N_VERTICES} самых надёжных вершин зоны {ROI}")
show_images(imgs[rank_g[:8]], titles=[f"{group_response[i]:+.2f}" for i in rank_g[:8]],
            suptitle=f"{ROI}: максимальный групповой отклик")
show_images(imgs[rank_g[-8:]], titles=[f"{group_response[i]:+.2f}" for i in rank_g[-8:]],
            suptitle=f"{ROI}: минимальный групповой отклик")

## 5. RDM — геометрия представлений

**RDM** (representational dissimilarity matrix) — матрица попарных расстояний между
паттернами ответа на разные картинки. Её ключевое свойство: размер зависит только от
числа стимулов, а не от числа вершин. Поэтому мозг с 19 065 вершинами и слой модели с
768 измерениями становятся сравнимыми.

Расстояние = `1 − корреляция Пирсона` по признакам. Такая метрика не меняется от того,
что весь ответ стал сильнее или слабее — а общая амплитуда fMRI гуляет от внимания и
усталости, и её попадание в геометрию нам не нужно.

In [ ]:
from scipy.spatial.distance import squareform

rdm = compute_rdm(patterns.astype(np.float64), metric="correlation")
full = squareform(rdm)

plt.figure(figsize=(5.2, 4.4))
plt.imshow(full[:200, :200], cmap="viridis")
plt.colorbar(label="1 − r")
plt.title(f"{SUBJECT} / {ROI}: RDM (первые 200 картинок)")
plt.xlabel("картинка"); plt.ylabel("картинка")
plt.tight_layout(); plt.show()

print(f"RDM: {len(rdm):,} пар, диапазон {rdm.min():.3f} .. {rdm.max():.3f}")

In [ ]:
# Какие две картинки мозг считает самыми ПОХОЖИМИ, а какие — самыми РАЗНЫМИ?
iu = np.triu_indices(len(imgs), k=1)
closest = np.argsort(rdm)[:4]
farthest = np.argsort(-rdm)[:4]

for name, pick in [("САМЫЕ ПОХОЖИЕ для мозга", closest), ("САМЫЕ РАЗНЫЕ для мозга", farthest)]:
    print(f"\n{name} ({ROI}):")
    for k in pick:
        a, b = imgs[iu[0][k]], imgs[iu[1][k]]
        show_images([a, b], cols=2, size=2.0, suptitle=f"расстояние {rdm[k]:.3f}")

Осторожно с интерпретацией: пары-рекордсмены сильно подвержены шуму — это экстремумы
распределения из 132 355 значений. Смотри на них как на иллюстрацию, а не доказательство.

## 6. Модель против мозга

Теперь берём RDM слоя модели и сравниваем с RDM зоны коры по Спирмену. Получается число:
насколько похожа геометрия.

**Ручка:** `MODEL` — любой из закэшированных энкодеров.

In [ ]:
import h5py

# ============ РУЧКИ ============
MODEL = "dinov2_vitb14_224"   # clip_vitb16_224 | vit_b16_224 | resnet50_224 | dinov2_vitl14_224
POOLING = "cls"               # для ViT: cls | patchmean;  для resnet50: gap
# ================================

path = ROOT / f"cache/activations/{MODEL}.h5"
with h5py.File(path, "r") as f:
    layers = sorted(k for k in f.keys() if k.endswith(f".{POOLING}"))
    model_rdms = {k: compute_rdm(f[k][:][IMAGES].astype(np.float64)) for k in layers}

scores = [compare_rdms(model_rdms[k], rdm) for k in layers]

plt.figure(figsize=(7, 3))
plt.plot(range(len(layers)), scores, "o-", color="#356")
plt.xticks(range(len(layers)), [k.split(".")[0] for k in layers], rotation=60, fontsize=7)
plt.ylabel("Spearman с RDM мозга"); plt.axhline(0, color="k", lw=0.5)
plt.title(f"{MODEL} ({POOLING}) против {ROI} у {SUBJECT}")
plt.tight_layout(); plt.show()

best = int(np.argmax(scores))
print(f"лучший слой: {layers[best]}  r = {scores[best]:.3f}")
print("\nЭто СЫРАЯ корреляция. В отчётах она делится на потолок шума — иначе зоны")
print("с разным качеством данных несравнимы. Потолки лежат в cache/rsa_results.json.")

In [ ]:
import json

ceilings = json.loads((ROOT / "cache/rsa_results.json").read_text())["ceilings"]
lo, hi = ceilings[ROI]
print(f"потолок для {ROI}: нижняя граница {lo:.3f}, верхняя {hi:.3f}")
print(f"лучший слой {MODEL}: сырое {scores[best]:.3f}  ->  нормированное {scores[best]/lo:.3f}")
print("\nТо есть модель забирает такую долю от объяснимой структуры.")

In [ ]:
# Где модель и мозг РАСХОДЯТСЯ сильнее всего?
from scipy.stats import rankdata

m = rankdata(model_rdms[layers[best]])
b = rankdata(rdm)
gap = m - b                     # >0: модель считает пару разной, мозг — похожей

for name, pick in [
    ("Модель разводит, мозг сближает", np.argsort(-gap)[:3]),
    ("Мозг разводит, модель сближает", np.argsort(gap)[:3]),
]:
    print(f"\n{name}:")
    for k in pick:
        show_images([imgs[iu[0][k]], imgs[iu[1][k]]], cols=2, size=2.0,
                    suptitle=f"ранг модели {int(m[k])}, ранг мозга {int(b[k])}")

## 7. Стек VLM

Здесь наш собственный вклад. Активации сняты со всего пути:
блоки зрительного энкодера → projector → все слои языковой модели, с позиций визуальных
токенов.

**Ручки:** `VLM` и `PROMPT`. Условие `none` — контроль без вопроса.

In [ ]:
import re

# ============ РУЧКИ ============
VLM = "smolvlm_500m"     # smolvlm_256m | smolvlm_500m
PROMPT = "none"          # none | objects | spatial | mood
# ================================

vpath = ROOT / f"cache/vlm_activations/{VLM}__{PROMPT}__before.h5"
with h5py.File(vpath, "r") as f:
    keys = [k for k in f.keys() if k != "image_index"]
    stored = f["image_index"][:]
    lookup = {int(x): i for i, x in enumerate(stored)}
    take = np.array([lookup[int(i)] for i in IMAGES])
    vision = sorted(k for k in keys if k.startswith("vision."))
    llm = sorted((k for k in keys if re.match(r"llm\.\d+\.imgmean$", k)),
                 key=lambda k: int(k.split(".")[1]))
    seq = vision + (["projector"] if "projector" in keys else []) + llm
    vlm_rdms = {k: compute_rdm(f[k][:][take].astype(np.float64)) for k in seq}

vlm_scores = [compare_rdms(vlm_rdms[k], rdm) / lo for k in seq]

plt.figure(figsize=(8, 3.2))
plt.plot(range(len(seq)), vlm_scores, "-", lw=1.8, color="#356")
plt.axvline(len(vision), color="k", ls=":", lw=1)
plt.text(len(vision), max(vlm_scores), " projector", fontsize=8, va="top")
plt.ylabel("RSA / потолок"); plt.axhline(0, color="k", lw=0.5)
plt.xlabel("позиция в стеке (блоки энкодера | projector | слои LLM)")
plt.title(f"{VLM}, промпт «{PROMPT}» против {ROI} у {SUBJECT}")
plt.tight_layout(); plt.show()

print("Смотри на две вещи: провал в середине зрительного энкодера (это гипотеза")
print("про токены-выбросы, стадия F1) и наклон на участке LLM.")

In [ ]:
# Сравнить все четыре условия задачи разом.
plt.figure(figsize=(8, 3.4))
for prompt in ["none", "objects", "spatial", "mood"]:
    p = ROOT / f"cache/vlm_activations/{VLM}__{prompt}__before.h5"
    if not p.exists():
        continue
    with h5py.File(p, "r") as f:
        st = f["image_index"][:]
        lk = {int(x): i for i, x in enumerate(st)}
        tk = np.array([lk[int(i)] for i in IMAGES])
        vals = [compare_rdms(compute_rdm(f[k][:][tk].astype(np.float64)), rdm) / lo for k in seq]
    plt.plot(range(len(seq)), vals, lw=1.5, label=prompt)
plt.axvline(len(vision), color="k", ls=":", lw=1)
plt.legend(fontsize=8); plt.ylabel("RSA / потолок"); plt.axhline(0, color="k", lw=0.5)
plt.title(f"{VLM}: меняет ли вопрос геометрию? ({ROI}, {SUBJECT})")
plt.tight_layout(); plt.show()

print("Блоки энкодера обязаны совпасть у всех условий: зрительная башня работает")
print("до языковой модели и вопроса не видит. Расходиться могут только слои LLM.")

## 8. Своё поле

Ниже — заготовка и список вопросов, на которые данные уже могут ответить. Всё нужное
загружено: `subj`, `patterns`, `imgs`, `rdm`, `IMAGES`, `VALID`, `STIMULI`, `DESIGN`.

**Что можно попробовать:**

1. **Насколько испытуемые похожи друг на друга?** Построй RDM одной зоны у двух разных
   людей и сравни. Это и есть потолок шума — верхняя граница того, что может объяснить
   модель.
2. **Отличается ли надёжность левого и правого полушария?** `subj.lh_size` делит вершины;
   посчитай `split_half_reliability` отдельно.
3. **Правда ли, что яркость картинки объясняет ранние зоны?** Посчитай среднюю яркость
   каждой картинки, сделай из неё RDM (расстояние = модуль разности) и сравни с `early`.
   У нас вышло 0.218 от потолка — проверь сам.
4. **Как влияет число усредняемых повторов?** Возьми `average_repeats` по одному повтору
   вместо трёх и посмотри, насколько упадёт корреляция с моделью.
5. **Где в коре модель работает лучше всего?** Пройди циклом по всем семи зонам с
   фиксированным слоем модели.
6. **Сколько картинок реально нужно?** Пересчитай RSA на 100, 200, 515 картинках и
   посмотри, когда оценка стабилизируется.

In [ ]:
# --- твоя песочница ---
# Пример к вопросу 1: похожи ли два человека между собой?

other = load_subject(ROOT / "data/betas/subj02_shared_betas.h5")
p2, _ = average_repeats(other, images=IMAGES, roi=ROI, valid=VALID)
rdm2 = compute_rdm(p2.astype(np.float64))

print(f"{ROI}: {SUBJECT} против subj02 — Spearman = {compare_rdms(rdm, rdm2):.3f}")
print(f"для сравнения, лучшая модель давала {scores[best]:.3f}")
print("\nЕсли человек с человеком сходится сильнее, чем модель с человеком —")
print("значит в данных есть структура, которую модель пока не улавливает.")

del other